# DFT XC skeleton 二阶导数分解 (TPSS0, MGGA)

该文档有 AI 辅助，但公式记号和程序记号比较混乱。我也作了很多修改，但思路还是没有理顺。

不过后面实现的算法，现在可以定型了。

我们有可能需要先实现一版 Python DFT 的实现，并且同时把公式思路理顺 (相当于构建一定程度的 harness，否则到时候过一两天，我写的和 AI 写的都认不清楚)。这样才能进一步开展 UKS 的实现，以及 Rust 下的迁移。

我们这边的算法看起来与 PySCF 有共通之处，但在一些根本问题上有差异。这体现在
- 对于 fxc 部分，我们直接使用 rho 导数，而不是将其转回到 AO 基表示。转回 AO 基表示其实代价是非常大的。
- 对于 vxc 部分的 non-diagonal 部分，有三种做法；我们采用第二种：
    1. (最直观做法，AI 做法) 直接使用 rho 导数，类似于 fxc 部分的处理。这里的问题是，我们需要生成 `[natm, 3, natm, 3, ngrids]` 的二阶密度导数矩阵来处理，尽管不全是不必要的计算、但会有很多碎片化的矩阵乘法。
    2. (当前策略) 类似于 `_get_vxc_diag`，先给出一个原子非依赖的 `[3, 3, nao, nao]` 的二阶分量矩阵，然后再依双原子加权求和。
    3. (PySCF 目前做法) 与 fxc 同时处理；在得到密度导数的基础上，对它转回到 AO 基表示，得到 `[natm, 3, 3, 3, nao, nao]` 的矩阵，然后再依单原子加权求和。这里的问题是，转回 AO 基表示的代价非常大，这种转回的成本可能相比于前面的策略增加了几倍。
- 总而言之，`_get_vxc_deriv2` 我认为是有优化空间的。也许这个函数在更高阶导数中会被用到，但在 Hessian 中我们有不同的处理方法。我相信在混合导数任务也是类似的。

DFT 麻烦就麻烦在它不完全是张量；它还要处理分量。厘清思路才能把代码写得清晰，或者至少能与文档产生确定的一一对应。这件事很意外，AI 其实更适合处理这种脏的代码；但我仍然偏好半古法 (半维新) 的方式来处理这个问题，目前完全交给 AI 托管似乎不现实，总得有人能理解。而我们作为科研程序员的工作，也是理解与实现的结合。

In [1]:
from pyscf import gto, dft, lib
import numpy as np
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
mf = dft.RKS(mol, xc="TPSS0").density_fit()
dat0 = np.load("nh3_r_tpss0.npz")
mf.mo_coeff = dat0["mo_coeff"]
mf.mo_occ = dat0["mo_occ"]
mf.mo_energy = dat0["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mocc = mo_coeff[:, mo_occ > 0]
occ_occupation = mo_occ[mo_occ > 0]
mocc_2 = np.einsum("pi,i->pi", mocc, occ_occupation**0.5)
dm0 = mocc @ mocc.T * 2
natm = mol.natm
nao = mol.nao
aoslices = mol.aoslice_by_atom()
ni = dft.numint.NumInt()

In [5]:
grids = dft.grid.Grids(mol)
grids.coords = coords = dat0["grid_coords"]
grids.weights = weights = dat0["grid_weights"]
ngrids = len(weights)

In [6]:
# Reference de_vxc from 06-1: this is what we want to reproduce.
de_ks_ref = np.load("nh3_r_tpss0_decomp.npz")["de_vxc"]
print("de_vxc_ref shape:", de_ks_ref.shape)
print("de_vxc_ref fp:   ", lib.fp(de_ks_ref))

de_vxc_ref shape: (4, 4, 3, 3)
de_vxc_ref fp:    -0.831082156811122


In [7]:
ao = ni.eval_ao(mol, grids.coords, deriv=3)
# ao_t = ao.swapaxes(-1, -2) # transposed AO, [t, u, g]
rho = ni.eval_rho2(mol, ao, mo_coeff, mo_occ, xctype="MGGA")
rho = rho[[0, 1, 2, 3, 5]]
rho.shape

(5, 43328)

In [8]:
xc_eff = ni.eval_xc_eff(mf.xc, rho, deriv=2, xctype="MGGA")
vxc = xc_eff[1]  # shape [5, ngrid]
fxc = xc_eff[2]  # shape [5, 5, ngrid]
print("vxc shape:", vxc.shape, "fxc shape:", fxc.shape)

vxc shape: (5, 43328) fxc shape: (5, 5, 43328)


In [9]:
TX, TY, TZ = 0, 1, 2
O = 0
X, Y, Z = 1, 2, 3
XX, XY, XZ = 4, 5, 6
YX, YY, YZ = 5, 7, 8
ZX, ZY, ZZ = 6, 8, 9
XXX, XXY, XXZ, XYY, XYZ, XZZ = 10, 11, 12, 13, 14, 15
YYY, YYZ, YZZ, ZZZ = 16, 17, 18, 19

In [10]:
ao_dm0 = ao @ dm0
ao_dm0.shape

(20, 43328, 49)

# fxc contribution

我感觉想不到更好的方案了。每一组 AO contraction 应该都是独一无二的。假使矩阵乘法的 FLOPs 是唯一的瓶颈，下面的代码我感觉已经是最优实现了。

In [11]:
# %%time
drho = np.zeros((natm, 3, 5, ngrids))
for A in range(natm):
    _, _, p0, p1 = aoslices[A]
    slc = slice(p0, p1)
    ao_slc = ao[:, :, slc]
    ao_dm0_slc = ao_dm0[:, :, slc]
    # components
    DERIV_COMPONENTS = [
        # RHO part
        [(TX, 0), (X, O)],
        [(TY, 0), (Y, O)],
        [(TZ, 0), (Z, O)],
        # SIGMA part (bra deriv 2)
        [(TX, X), (XX, O)],
        [(TX, Y), (XY, O)],
        [(TX, Z), (XZ, O)],
        [(TY, X), (YX, O)],
        [(TY, Y), (YY, O)],
        [(TY, Z), (YZ, O)],
        [(TZ, X), (ZX, O)],
        [(TZ, Y), (ZY, O)],
        [(TZ, Z), (ZZ, O)],
        # SIGMA part (bra deriv 1, ket deriv 1)
        [(TX, X), (X, X)],
        [(TX, Y), (X, Y)],
        [(TX, Z), (X, Z)],
        [(TY, X), (Y, X)],
        [(TY, Y), (Y, Y)],
        [(TY, Z), (Y, Z)],
        [(TZ, X), (Z, X)],
        [(TZ, Y), (Z, Y)],
        [(TZ, Z), (Z, Z)],
        # TAU part
        [(TX, 4), (XX, X)],
        [(TX, 4), (XY, Y)],
        [(TX, 4), (XZ, Z)],
        [(TY, 4), (YX, X)],
        [(TY, 4), (YY, Y)],
        [(TY, 4), (YZ, Z)],
        [(TZ, 4), (ZX, X)],
        [(TZ, 4), (ZY, Y)],
        [(TZ, 4), (ZZ, Z)],
    ];
    
    for ((t, v), (cbra, cket)) in DERIV_COMPONENTS:
        # drho[A, t, v] += np.einsum("gu, gu -> g", ao_slc[cbra], ao_dm0_slc[cket])
        drho[A, t, v] -= np.einsum("gu, gu -> g", ao_slc[cbra], ao_dm0_slc[cket])
# scale symmetric coeff: RHO and SIGMA (0..3) get *2, TAU (4) does not
drho[:, :, :4] *= 2

In [12]:
lib.fp(drho)

np.float64(-16948168.18739754)

In [13]:
de_fxc = np.einsum("g, Atxg, xyg, Bsyg -> ABts", weights, drho, fxc, drho)
print(lib.fp(de_fxc))

-29.390069496788136


In [14]:
# --- dao_vxc_diag --- #

dao_vxc_diag = np.zeros((6, nao))  # 6 denotes xx, xy, xz, yy, yz, zz
wv = weights * vxc  # [5, ngrids]

# Contribution 1: ao[i+4]^T @ (wv[0]*ao[0] + wv[1]*ao[1] + wv[2]*ao[2] + wv[3]*ao[3])
aow_diag = (np.einsum("gu, g -> gu", ao_dm0[0], wv[0])
          + np.einsum("gu, g -> gu", ao_dm0[1], wv[1])
          + np.einsum("gu, g -> gu", ao_dm0[2], wv[2])
          + np.einsum("gu, g -> gu", ao_dm0[3], wv[3]))
for idx, its in enumerate([XX, XY, XZ, YY, YZ, ZZ]):
    dao_vxc_diag[idx] += 2 * np.einsum("gu, gu -> u", ao[its], aow_diag)

# Contribution 2 (GGA triple-derivative part):
# (wv[1]*ao[triple1] + wv[2]*ao[triple2] + wv[3]*ao[triple3])^T @ ao[0]
TRIPLE_DERIV_DIAG = [
    [XXX, XXY, XXZ],  # xx
    [XXY, XYY, XYZ],  # xy
    [XXZ, XYZ, XZZ],  # xz
    [XYY, YYY, YYZ],  # yy
    [XYZ, YYZ, YZZ],  # yz
    [XZZ, YZZ, ZZZ],  # zz
]
for idx, (i3x, i3y, i3z) in enumerate(TRIPLE_DERIV_DIAG):
    aow_triple = (np.einsum("gu, g -> gu", ao[i3x], wv[1])
                + np.einsum("gu, g -> gu", ao[i3y], wv[2])
                + np.einsum("gu, g -> gu", ao[i3z], wv[3]))
    dao_vxc_diag[idx] += 2 * np.einsum("gu, gu -> u", aow_triple, ao_dm0[0])

# Contribution 3 (TAU part): ao[triple_idx]^T @ (wv_tau[4] * ao[direction])
aow_diag_tau = [np.einsum("gu, g -> gu", ao_dm0[d], wv[4]) for d in [X, Y, Z]]

TAU_DIAG_COMPONENTS = [
    ([XXX, XXY, XXZ, XYY, XYZ, XZZ], 0),  # direction x: ao[triple]^T @ aow_tau_x
    ([XXY, XYY, XYZ, YYY, YYZ, YZZ], 1),  # direction y
    ([XXZ, XYZ, XZZ, YYZ, YZZ, ZZZ], 2),  # direction z
]
for i, (triple_indices, direction) in enumerate(TAU_DIAG_COMPONENTS):
    for idx, j in enumerate(triple_indices):
        dao_vxc_diag[idx] += np.einsum("gu, gu -> u", ao[j], aow_diag_tau[direction])

de_vxc_diag = np.zeros((natm, natm, 6))
for A in range(natm):
    _, _, p0A, p1A = aoslices[A]
    slcA = slice(p0A, p1A)
    de_vxc_diag[A, A] += np.einsum("Au -> A", dao_vxc_diag[:, slcA])
de_vxc_diag = de_vxc_diag[:, :, [[0, 1, 2], [1, 3, 4], [2, 4, 5]]]

print("de_vxc_diag fp:", lib.fp(de_vxc_diag))

de_vxc_diag fp: 44.68386358957387


In [15]:
# --- dao_vxc_diag --- #

dao_vxc_diag = np.zeros((6, nao, nao))  # 6 denotes xx, xy, xz, yy, yz, zz

# Contribution 1: ao[i+4]^T @ (wv[0]*ao[0] + wv[1]*ao[1] + wv[2]*ao[2] + wv[3]*ao[3])
aow_diag = (np.einsum("gu, g -> gu", ao[0], wv[0])
          + np.einsum("gu, g -> gu", ao[1], wv[1])
          + np.einsum("gu, g -> gu", ao[2], wv[2])
          + np.einsum("gu, g -> gu", ao[3], wv[3]))
for idx, its in enumerate([XX, XY, XZ, YY, YZ, ZZ]):
    dao_vxc_diag[idx] += 2 * ao[its].T @ aow_diag

# Contribution 2 (GGA triple-derivative part):
# (wv[1]*ao[triple1] + wv[2]*ao[triple2] + wv[3]*ao[triple3])^T @ ao[0]
TRIPLE_DERIV_DIAG = [
    [XXX, XXY, XXZ],  # xx
    [XXY, XYY, XYZ],  # xy
    [XXZ, XYZ, XZZ],  # xz
    [XYY, YYY, YYZ],  # yy
    [XYZ, YYZ, YZZ],  # yz
    [XZZ, YZZ, ZZZ],  # zz
]
for idx, (i3x, i3y, i3z) in enumerate(TRIPLE_DERIV_DIAG):
    aow_triple = (np.einsum("gu, g -> gu", ao[i3x], wv[1])
                + np.einsum("gu, g -> gu", ao[i3y], wv[2])
                + np.einsum("gu, g -> gu", ao[i3z], wv[3]))
    dao_vxc_diag[idx] += 2 * aow_triple.T @ ao[0]

# Contribution 3 (TAU part): ao[triple_idx]^T @ (wv_tau[4] * ao[direction])
aow_diag_tau = [np.einsum("gu, g -> gu", ao[d], wv[4]) for d in [X, Y, Z]]

TAU_DIAG_COMPONENTS = [
    ([XXX, XXY, XXZ, XYY, XYZ, XZZ], 0),  # direction x: ao[triple]^T @ aow_tau_x
    ([XXY, XYY, XYZ, YYY, YYZ, YZZ], 1),  # direction y
    ([XXZ, XYZ, XZZ, YYZ, YZZ, ZZZ], 2),  # direction z
]
for i, (triple_indices, direction) in enumerate(TAU_DIAG_COMPONENTS):
    for idx, j in enumerate(triple_indices):
        dao_vxc_diag[idx] += ao[j].T @ aow_diag_tau[direction]

de_vxc_diag = np.zeros((natm, natm, 6))
for A in range(natm):
    _, _, p0A, p1A = aoslices[A]
    slcA = slice(p0A, p1A)
    de_vxc_diag[A, A] += np.einsum("Auv, uv -> A", dao_vxc_diag[:, slcA], dm0[slcA])
de_vxc_diag = de_vxc_diag[:, :, [[0, 1, 2], [1, 3, 4], [2, 4, 5]]]

print("de_vxc_diag fp:", lib.fp(de_vxc_diag))

de_vxc_diag fp: 44.683863589573576


In [16]:
# --- dao_vxc --- #

wv = weights * vxc  # [5, ngrids]
dao_vxc = np.zeros((3, 3, nao, nao))

# GGA part (RHO + SIGMA)

GGA_CALLS = [[XX, XY, XZ], [YX, YY, YZ], [ZX, ZY, ZZ]]

aowv = [None, None, None]
for t in range(3):
    aowv[t] = 0.5 * np.einsum("gu, g -> gu", ao[t + 1], wv[0])
    for r in range(3):
        aowv[t] += np.einsum("gu, g -> gu", ao[GGA_CALLS[t][r]], wv[r + 1])

for t in range(3):
    for s in range(3):
        dao_vxc[t, s] += 2 * aowv[s].T @ ao[t + 1]     # ipip[t,s]

# TAU part: ipip from three _d1d2_dot_ calls with wv[4] *= 0.25
# aow_tau[i] = 0.25 * wv_tau[4] * ao[4+i]  for i=0..5 (XX..ZZ)
aowv = [np.einsum("gu, g -> gu", ao[4 + i], wv[4]) for i in range(6)]

TAU_CALLS = [
    ([0, 1, 2], [XX, XY, XZ]),  # {aow_tau[XX], aow_tau[XY], aow_tau[XZ]} vs {ao[XX], ao[XY], ao[XZ]}
    ([1, 3, 4], [YX, YY, YZ]),  # {aow_tau[XY], aow_tau[YY], aow_tau[YZ]} vs {ao[YX], ao[YY], ao[YZ]}
    ([2, 4, 5], [ZX, ZY, ZZ]),  # {aow_tau[XZ], aow_tau[YZ], aow_tau[ZZ]} vs {ao[ZX], ao[ZY], ao[ZZ]}
]

dao_vxc_tau = np.zeros((3, 3, nao, nao))

for r_bra, r_ket in TAU_CALLS:
    for t in range(3):
        for s in range(t + 1):
            dao_vxc_tau[t, s] += 0.5 * aowv[r_bra[s]].T @ ao[r_ket[t]]    # ipip[d1,d2]

for t in range(3):
    for s in range(t):
        dao_vxc_tau[s, t] = dao_vxc_tau[t, s].T

dao_vxc += dao_vxc_tau
dao_vxc += dao_vxc.transpose(1, 0, 3, 2)  # [s,t] with AO indices transposed

de_vxc = np.zeros((natm, natm, 3, 3))
for A in range(natm):
    _, _, p0A, p1A = aoslices[A]
    slcA = slice(p0A, p1A)
    for B in range(A + 1):
        _, _, p0B, p1B = aoslices[B]
        slcB = slice(p0B, p1B)
        de_vxc[A, B] += np.einsum("tsuv, uv -> ts", dao_vxc[:, :, slcB, slcA], dm0[slcB, slcA])
        if A != B:
            de_vxc[B, A] = de_vxc[A, B].T
print("de_vxc fp:", lib.fp(de_vxc))

de_vxc fp: -16.124876249597484


In [17]:
de_xc_recap = de_vxc_diag + de_vxc + de_fxc
assert np.allclose(de_xc_recap, de_ks_ref)

In [18]:
dat = dict(np.load("nh3_r_tpss0_decomp.npz"))
dat.update({
    "de_vxc_diag": de_vxc_diag,
    "de_vxc": de_vxc,
    "de_fxc": de_fxc,
})
np.savez("nh3_r_tpss0_decomp.npz", **dat)